# Introducción a redes neuronales.

![](https://i.imgur.com/ktoS9gT.png)

## CdeCMx 2025 - GTO2: Cosmic distortions.

### By: Gabriel Missael Barco & Zaid de Anda Mariscal

*Parte de este notebook está adaptado del repositorio [PyTorch Image Classification, by Ben Trevett](https://github.com/bentrevett/pytorch-image-classification)😃. Los invitamos a revisar el resto del contenido!*

Las tareas que realizaremos en este club son:
1. **Clasificación**: Distinguir entre diferentes tipos de objetos. Por ejemplo, una galaxia normal de un lente gravitacional, o dígitos escritos a mano.
2. **Regresión**: Predecir el valor de algún parámetro de un sistema. Por ejemplo, el radio de einstein de un lente gravitacional.

Usaremos **PyTorch**, un paquete open-source en python para crear y entrenar redes neuronales, y comenzaremos el día de hoy con la arquitectura más básica: Percetprones de capas multiples (multilayer perceptrons). Mañana, mejoraremos nuestra red para incluir una red neuronal convolucional!.

Primero, entrenaremos un clasificador de digitos escritos a mano del 0 al 9, con el famoso dataset MNIST. Después, aplicaremos arquitecturas similares para analizar lentes gravitacionales!.


### Procesamiento inicial de datos.

Los módulos que usaremos en este notebook son los siguientes:

- torch para funcionalidades generales.
- torch.nn and torch.nn.functional para funcionoes para las redes neuronales.
- torch.optim para entrenar nuestros modelos!
- torch.utils.data para crear los datasets de entrenamiento.
- torchvision.transforms para "aumentar" nuestro dataset con transformaciones.
- Métricas de sklearn para visualizar como se comporta nuestro modelo.
- Visualización de "manifold" de sklearn para entender que es lo que la red neuronal está aprendiendo.
- matplotlib para crear gráficas.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.utils.data as data

import torchvision.transforms as transforms
import torchvision.datasets as datasets

from sklearn import metrics
from sklearn import decomposition
from sklearn import manifold
from tqdm.notebook import trange, tqdm
import matplotlib.pyplot as plt
import numpy as np

import copy
import random
import time

Para asegurarnos de que los resultados son "reproducibles", ponemos una semilla de aleatoriedad ("random seed").

In [ ]:
SEED = 1234

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

Primero, descargamos el dataset de MNIST:

In [ ]:
ROOT = '.data'

train_data = datasets.MNIST(root=ROOT,
                            train=True,
                            download=True)

Es buena práctica normalizar nuestros datos, ya que es mas sencillo para la red neuronal aprender cuando los datos se encuentran dentro de cierto rango. Esto normalmente significa que tenga un promedio de cero, y una desviación estandar de 1.

Para ello, calculamos el promedio $\mu$, y la desviación estandar $\sigma$, y normalizamos:

$$ \bar{x} = \frac{x - \mu}{\sigma}$$

**Nota**: Es importante calcular $\mu$ y $\sigma$ solamente con el set de entrenamiento!.

En particular, nuestros datos están en el rango de 0 a 255, por lo que dividimos entre 255 para que los datos estén entre 0 y 1.

In [ ]:
mean = train_data.data.float().mean() / 255
std = train_data.data.float().std() / 255

In [ ]:
print(f'Calculated mean: {mean}')
print(f'Calculated std: {std}')

Ahora que tenemos nuestros datos, podemos realizar "aumentaciones", que son transformaciones a nuestros datos que *en principio* no cambian la clase o valor que queremos predecir, y que además, se podrían encontrar de manera natural en nuestro dataset. Por ejemplo, los digitos podrían estar un poco recorridos arriba o abajo, pero no escritos de cabeza!

Para ayudar a que nuestro modelo aprenda estas simetrias o propiedades esperadas de nuestros datos, podemos incluirlas en las transformaciones de los datos. Esto, además, ayuda a tener *mas datos de entrenamiento*, que generalmente ayuda a tener mejores modelos.


Algunas transformaciones que usaremos son:
- `RandomRotation` - Rota de manera aleatoria los digitos por $x = \pm 5$ grados (muy poco, algo que podría pasar con dígitos!). Las galaxias y los lentes gravitacionales tienen simetria de rotación completa, por lo que pueden tener cualquier angulo aquí.
- `RandomCrop` - Añadimos un pequeño borde de 2 pixeles a las imagenes (conocido como *padding**), para después cortar de manera aleatoria un segmento de $28x28$, que es la resolución original de los datos.
- `ToTensor()` - Esto convierte las imagenes a tensores (o matrices), lo cual es necesario para usarlo en pytorch).
- `Normalize` - Esto realiza la normalización de los datos que mencionamos anteriormente!.

Estas transformación solamente se aplican al set de entrenamiento, mientras el test set (de evaluación), se deja sin tranformar.


In [ ]:
train_transforms = transforms.Compose([
                            transforms.RandomRotation(5, fill=(0,)),
                            transforms.RandomCrop(28, padding=2),
                            transforms.ToTensor(),
                            transforms.Normalize(mean=[mean], std=[std])
                                      ])

test_transforms = transforms.Compose([
                           transforms.ToTensor(),
                           transforms.Normalize(mean=[mean], std=[std])
                                     ])

Cargamos los datos y asignamos las transformaciones correspondientes.

In [ ]:
train_data = datasets.MNIST(root=ROOT,
                            train=True,
                            download=True,
                            transform=train_transforms)

test_data = datasets.MNIST(root=ROOT,
                           train=False,
                           download=True,
                           transform=test_transforms)

Tamaño de los sets:

In [ ]:
print(f'Número de ejemplos de entrenamiento: {len(train_data)}')
print(f'Número de ejemplos de test: {len(test_data)}')

Aquí visualizamos algunos ejemplos de los dígitos

In [ ]:
def plot_images(images):

    n_images = len(images)

    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))

    fig = plt.figure(dpi = 150)
    for i in range(rows*cols):
        ax = fig.add_subplot(rows, cols, i+1)
        ax.imshow(images[i].view(28, 28).cpu().numpy(), cmap='bone')
        ax.axis('off')

Mostramos 25 imagenes del test de entrenamiento, las cuales ya tendrán las transformaciones que definimos.

Siempre siempre siempre es una buena práctica crear muchas gráficas y visualizaciones de lo que estés haciendo para asegurarte de que tiene sentido!

In [ ]:
N_IMAGES = 50

images = [image for image, label in [train_data[i] for i in range(N_IMAGES)]]

plot_images(images)

El dataset de MNIST no tiene un set de validación, que es el que usamos durante la definición del modelo para seleccionar hiperparametros, cómo número de capas, tipo de neuronas, etc. No podemos usar el test set para esto ya que se debe usar **únicamente para evaluación** al final de definir todo y entrenar nuestro modelo.

Por esto, tomaremos un pedazo del set de entrenamiento para usarlo como set de validación: no para entrenar, pero si para evaluar el módelo mientras tomamos decisiones.

In [ ]:
VALID_RATIO = 0.9

n_train_examples = int(len(train_data) * VALID_RATIO)
n_valid_examples = len(train_data) - n_train_examples

In [ ]:
train_data, valid_data = data.random_split(train_data,
                                           [n_train_examples, n_valid_examples])

El tamaño final de nuestros sets de entrenamiento, evaluación, y validación:

In [ ]:
print(f'Number of training examples: {len(train_data)}')
print(f'Number of validation examples: {len(valid_data)}')
print(f'Number of testing examples: {len(test_data)}')

Las transformaciones del set de validación deben de ser las mismas que las del test set:

In [ ]:
valid_data = copy.deepcopy(valid_data)
valid_data.dataset.transform = test_transforms

Ahora definiremos los `DataLoader` para cada uno de los training/validation/test sets que crearemos. Estos objectos son útiles ya que nos permiten *iterar* sobre ellos, lo cual será necesario para el enternamiento del modelo.

Sólamente el set de entrenamieneto necesita ser *revuelto* (shuffled), para cambiar los ejemplos que usa la red neuronal en cada época.

Finalmente, usaremos *descenso del gradiente estocastico*, que discutimos en el pizarrón. El tamaño del batch size es generalmente tan grande como sea posible de acuerdo al cómputo al que tengamos acceso.

In [ ]:
BATCH_SIZE = 512

train_iterator = data.DataLoader(train_data,
                                 shuffle=True,
                                 batch_size=BATCH_SIZE)

valid_iterator = data.DataLoader(valid_data,
                                 batch_size=BATCH_SIZE)

test_iterator = data.DataLoader(test_data,
                                batch_size=BATCH_SIZE)

### Definiendo el módelo.

Nuestro módelo sera un multilayer perceptron (MLP) con dos capas octultas, cómo se muestra en la imágen:

![](https://github.com/bentrevett/pytorch-image-classification/blob/master/assets/mlp-mnist.png?raw=1)

Para ello, dado que los MLPs trabajan con vectores (y no matrices), debemos aplanar la imagen de 28x28 en un vector de 784 elementos o *features*.

Posteriormente, las capas ocultas tendrán 250 y 100 neuronas, para reducir la dimensionalidad de la representación de los dígitos, y la ultima capa (de salida) tendrá 10 neuronas, cada una representando la *probabilidad* de que la imagén sea un dado dígito entre 0 y 9.

Las transformaciones $784 \to 250 \to 100 \to 10$ son realizadas for capas *lineares*, donde cada neurona de una capa está *conectada* con todas las neuronas de la capa anterior, donde cada conexión tiene cierta *fuerza*, representada por un parámetro con un cierto valor númerico. Estas *fuerzas de conexión* son precisamente los párametros de la red neuronal, y los cuales optimizaremos para que la red neuronal funcione!

Después de la transformación lineal, se aplica una función *no lineal* conocida como función de activación, esto para cada neurona. Este elemento es **muy** importante, ya que es lo que le permite a la red neuronal aprender relaciones complicadas! Usamos en particular la función de activación ReLU (rectified linear unit), que se define como se muestra en la imágen:

![](https://github.com/bentrevett/pytorch-image-classification/blob/master/assets/relu.png?raw=1)

Por qué elegimos la arquitectura $784 \to 250 \to 100 \to 10$? Hay alguna receta mágica para crear una arquitectura que funcione para cualquier problema? No! Cada problema es diferente, requiriendo modelos mas o menos grandes, y con tipos de neuronas y arquitecturas mas complejas.

In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim, output_dim):
        super().__init__()

        self.input_fc = nn.Linear(input_dim, 250)
        self.hidden_fc = nn.Linear(250, 100)
        self.output_fc = nn.Linear(100, output_dim)

    def forward(self, x):

        # x = [batch size, height, width]

        batch_size = x.shape[0]

        x = x.view(batch_size, -1)

        # x = [batch size, height * width]

        h_1 = F.relu(self.input_fc(x))

        # h_1 = [batch size, 250]

        h_2 = F.relu(self.hidden_fc(h_1))

        # h_2 = [batch size, 100]

        y_pred = self.output_fc(h_2)

        # y_pred = [batch size, output dim]

        return y_pred, h_2

Definimos una instancia de nuestro modelo:

In [ ]:
INPUT_DIM = 28 * 28
OUTPUT_DIM = 10

model = MLP(INPUT_DIM, OUTPUT_DIM)

Veámos cuantos parámetros tiene nuestro modelo!

In [ ]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [ ]:
print(f'El módelo tiene {count_parameters(model):,} parámetros')

### Entrenar el modelo

Ahora, vamos a definir el ciclo de entrenamiento, en el cuál actualizaremos los parámetros de la red neuronal para que pueda identificar dígitos.

Para ello, necesitamos definir nuestra función de perdida (loss function). Esta función, que es al que vamos a optimizar, depende del problema a resolver y de nuestro tipo de datos. En general, tiene una interpretación probabilistica, pero no iremos en mucho detalle. Las loss functions que usaremos son:
- Cross-entropy para clasificación
-  Mean-squared-error para regresión.

En general, los pasos a seguir en cada iteración son:
1. Pasar un bacth de datos por el modelo
2. Comparar la respuesta del modelo con la clase real
3. Calcualr el *gradiente* de cada uno de los parametros con respecto a la función de costo.
4. Actualizar el valor de cáda parametro restando el valor del gradiente multiplicado por el ritmo de aprendizaje *learning rate*. La actualización para el parámetro $i$ es:

$$w_i = w_i - \ell \times \frac{\partial L}{\partial w_i}$$

Utilizaremos una versión más complicada y eficiente de este procedimiento, conocido cómo el algoritmo *Adam*.

In [ ]:
optimizer = optim.Adam(model.parameters(), lr=0.0001)

Primero, para clasificar los dígitos, utilizaremos la
`CrossEntropyLoss`, que primero calcula una función de activación *softmax* (para convertir el output en probabilidades entre 0 y 1, cómo con un dado con 10 caras):

$$\text{softmax }(\mathbf{x}) = \frac{e^{x_i}}{\sum_j e^{x_j}}$$

Y luego calcula la función de costo. Entonces, el objetivo es básicamente que la red neuronal le asigne la probabilidad más alta a la clase correcta.


In [ ]:
criterion = nn.CrossEntropyLoss()

Usamos la GPU que nos provee Google Collaboratory:

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [ ]:
model = model.to(device)
criterion = criterion.to(device)

Esta función nos ayuda a evaluar el modelo contando cuantas veces se le asigna la mayor probabilidad a la clase correcta.

In [ ]:
def calculate_accuracy(y_pred, y):
    top_pred = y_pred.argmax(1, keepdim=True)
    correct = top_pred.eq(y.view_as(top_pred)).sum()
    acc = correct.float() / y.shape[0]
    return acc

Finalmente, creamos nuestro ciclo de entrenamiento. Para ello:
1. Ponemos nuestro modelo en modo de entrenamiento.
2. Iteramos sobre nuestro dataloader de entrenamiento, y para cada batch:
    1. Colocamos el batch en la GPU
    2. Pasamos el batch de las imágenes por el modelo para obtener las predicciones.
    3. Calculamos la función de costo usando la predicción y la clase real.
    4. Calculamos el progreso del modelo con el accuracy
    5. Calculamos el gradiente para cada parámetro.
    6. Actualizamos los parámetros.
    7. Actualizamos las métricas.
    8. Limpiamos los gradientes utilizados.

In [ ]:
def train(model, iterator, optimizer, criterion, device):

    epoch_loss = 0
    epoch_acc = 0

    model.train()

    for (x, y) in tqdm(iterator, desc="Training", leave=False):

        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        y_pred, _ = model(x)

        loss = criterion(y_pred, y)

        acc = calculate_accuracy(y_pred, y)

        loss.backward()

        optimizer.step()

        epoch_loss += loss.item()
        epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

El ciclo de evaluación del modelo es similar, pero poniendo el modelo en modo evaluación, apagando los gradientes, y no actualizando parámetros ni usando el optimizador.

In [ ]:
def evaluate(model, iterator, criterion, device):

    epoch_loss = 0
    epoch_acc = 0

    model.eval()

    with torch.no_grad():

        for (x, y) in tqdm(iterator, desc="Evaluating", leave=False):

            x = x.to(device)
            y = y.to(device)

            y_pred, _ = model(x)

            loss = criterion(y_pred, y)

            acc = calculate_accuracy(y_pred, y)

            epoch_loss += loss.item()
            epoch_acc += acc.item()

    return epoch_loss / len(iterator), epoch_acc / len(iterator)

In [ ]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [ ]:
EPOCHS = 5

best_valid_loss = float('inf')

for epoch in trange(EPOCHS):

    start_time = time.monotonic()

    train_loss, train_acc = train(model, train_iterator, optimizer, criterion, device)
    valid_loss, valid_acc = evaluate(model, valid_iterator, criterion, device)

    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'tut1-model.pt')

    end_time = time.monotonic()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\tTrain Loss: {train_loss:.3f} | Train Acc: {train_acc*100:.2f}%')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. Acc: {valid_acc*100:.2f}%')

Después del entrenamiento, tomamos el modelo con la mejor loss de validación, y lo usamos para ver su comportamiento en el test set.

In [ ]:
model.load_state_dict(torch.load('tut1-model.pt'))

test_loss, test_acc = evaluate(model, test_iterator, criterion, device)

In [ ]:
print(f'Test Loss: {test_loss:.3f} | Test Acc: {test_acc*100:.2f}%')

Obtenemos un accuracy muy buena! Casi todos los dígitos son clasificados de manera correcta 😀

Este performance se puede mejorar con un mejor modelo, mas datos, mas tiempo de entrenamiento, etc.

### Examinando el modelo

In [ ]:
def get_predictions(model, iterator, device):

    model.eval()

    images = []
    labels = []
    probs = []

    with torch.no_grad():

        for (x, y) in iterator:

            x = x.to(device)

            y_pred, _ = model(x)

            y_prob = F.softmax(y_pred, dim=-1)

            images.append(x.cpu())
            labels.append(y.cpu())
            probs.append(y_prob.cpu())

    images = torch.cat(images, dim=0)
    labels = torch.cat(labels, dim=0)
    probs = torch.cat(probs, dim=0)

    return images, labels, probs

In [ ]:
images, labels, probs = get_predictions(model, test_iterator, device)

pred_labels = torch.argmax(probs, 1)

In [ ]:
def plot_confusion_matrix(labels, pred_labels):

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(1, 1, 1)
    cm = metrics.confusion_matrix(labels, pred_labels)
    cm = metrics.ConfusionMatrixDisplay(cm, display_labels=range(10))
    cm.plot(values_format='d', cmap='viridis', ax=ax)

La matriz de confusión nos ayuda a ver cuales digitos son clasificados de manera correcta, y cuales son los que mas se confunden (por ejemplo, los dígitos 9 y 4 se confunden mucho!)

In [ ]:
plot_confusion_matrix(labels, pred_labels)

Veamos algúnos ejemplos clasificados erroneamente!



In [ ]:
corrects = torch.eq(labels, pred_labels)

In [ ]:
incorrect_examples = []

for image, label, prob, correct in zip(images, labels, probs, corrects):
    if not correct:
        incorrect_examples.append((image, label, prob))

incorrect_examples.sort(reverse=True,
                        key=lambda x: torch.max(x[2], dim=0).values)

In [ ]:
def plot_most_incorrect(incorrect, n_images):

    rows = int(np.sqrt(n_images))
    cols = int(np.sqrt(n_images))

    fig = plt.figure(figsize=(20, 10))
    for i in range(rows*cols):
        ax = fig.add_subplot(rows, cols, i+1)
        image, true_label, probs = incorrect[i]
        true_prob = probs[true_label]
        incorrect_prob, incorrect_label = torch.max(probs, dim=0)
        ax.imshow(image.view(28, 28).cpu().numpy(), cmap='bone')
        ax.set_title(f'true label: {true_label} ({true_prob:.3f})\n'
                     f'pred label: {incorrect_label} ({incorrect_prob:.3f})')
        ax.axis('off')
    fig.subplots_adjust(hspace=0.5)

In [ ]:
N_IMAGES = 25

plot_most_incorrect(incorrect_examples, N_IMAGES)

Finalmente, podemos crear representaciones de baja dimensionalidad para observar si los digitos se separan con éxito!

In [ ]:
def get_representations(model, iterator, device):

    model.eval()

    outputs = []
    intermediates = []
    labels = []

    with torch.no_grad():

        for (x, y) in tqdm(iterator):

            x = x.to(device)

            y_pred, h = model(x)

            outputs.append(y_pred.cpu())
            intermediates.append(h.cpu())
            labels.append(y)

    outputs = torch.cat(outputs, dim=0)
    intermediates = torch.cat(intermediates, dim=0)
    labels = torch.cat(labels, dim=0)

    return outputs, intermediates, labels

Obtenemos la representación:

In [ ]:
outputs, intermediates, labels = get_representations(model,
                                                     train_iterator,
                                                     device)

The data we want to visualize is in ten dimensions and 100 dimensions. We want to get this down to two dimensions, so we can actually plot it.

The first technique we'll use is PCA (principal component analysis). First, we'll define a function to calculate the PCA of our data, and then we'll define a function to plot it.

In [ ]:
def get_pca(data, n_components=2):
    pca = decomposition.PCA()
    pca.n_components = n_components
    pca_data = pca.fit_transform(data)
    return pca_data

In [ ]:
def plot_representations(data, labels, n_images=None):
    if n_images is not None:
        data = data[:n_images]
        labels = labels[:n_images]
    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111)
    scatter = ax.scatter(data[:, 0], data[:, 1], c=labels, cmap='tab10')
    handles, labels = scatter.legend_elements()
    ax.legend(handles=handles, labels=labels)

La representación en 2 dimensiones del output de las 10 clases:

In [ ]:
output_pca_data = get_pca(outputs)
plot_representations(output_pca_data, labels)

Ahora, el output de la última capa oculta, de $100$ a $2$ dimensiones

In [ ]:
intermediate_pca_data = get_pca(intermediates)
plot_representations(intermediate_pca_data, labels)

¿Qué pasa si intentamos generar un dígito falso?

In [ ]:
def imagine_digit(model, digit, device, n_iterations=50_000):

    model.eval()

    best_prob = 0
    best_image = None

    with torch.no_grad():

        for _ in trange(n_iterations):

            x = torch.randn(256, 28, 28).to(device)

            y_pred, _ = model(x)

            preds = F.softmax(y_pred, dim=-1)

            _best_prob, index = torch.max(preds[:, digit], dim=0)

            if _best_prob > best_prob:
                best_prob = _best_prob
                best_image = x[index]

    return best_image, best_prob

In [ ]:
fig, axs = plt.subplots(1, 5, figsize=(10, 2))
for i in range(5):
    x = torch.randn(28, 28).to(device) + 0.5
    axs[i].imshow(x.cpu().numpy(), cmap='bone')
    axs[i].axis('off')

Veamos si podemos generar, aleatoriamente, el número 3!

In [ ]:
DIGIT = 3

best_image, best_prob = imagine_digit(model, DIGIT, device)

Veamos el puntaje mas alto que obtenemos!

In [ ]:
print(f'Best image probability: {best_prob.item()*100:.2f}%')

¿Creen que se vaya parecer al número 3?

In [ ]:
plt.imshow(best_image.cpu().numpy(), cmap='bone')
plt.axis('off');

Finalmente, podemos visualizar los valores de los parámetros. Se ven figuras interesantes, pero en realidad no tiene un significado en particular (al menos para el caso de MLPs)

In [ ]:
def plot_weights(weights, n_weights):

    rows = int(np.sqrt(n_weights))
    cols = int(np.sqrt(n_weights))

    fig = plt.figure(figsize=(20, 10))
    for i in range(rows*cols):
        ax = fig.add_subplot(rows, cols, i+1)
        ax.imshow(weights[i].view(28, 28).cpu().numpy(), cmap='bone')
        ax.axis('off')

In [ ]:
N_WEIGHTS = 25

weights = model.input_fc.weight.data

plot_weights(weights, N_WEIGHTS)

# Reto: Clasificador de lentes vs galaxias regulares 😀

In [ ]:
import gdown
import numpy as np

file_id = "1URDQ0HoVC9IEGQxvjpqhPDDb0-_wwpa7"
url = f"https://drive.google.com/uc?id={file_id}"
output = "labels.pt"
gdown.download(url, output, quiet=False)

labels = torch.load("/content/labels.pt")

In [ ]:
file_id = "1BdZ-ttOES9xGxs1zToPh0ReMtkcGONjN"
url = f"https://drive.google.com/uc?id={file_id}"
output = "dataset.pt"
gdown.download(url, output, quiet=False)

dataset = torch.load("/content/dataset.pt")

In [ ]:
# Plot a grid
fig, axs = plt.subplots(4, 8, figsize=(20, 10))
for i, ax in enumerate(axs.flat):
    ax.imshow(dataset[i], origin='lower')
    ax.axis('off')
    title = "Lente" if labels[i].item() == 1 else "Galaxia"
    ax.set_title(title)
plt.subplots_adjust(wspace=0.2, hspace=0.2)
plt.show()